# Euro Area Housing & Monetary Policy: Panel Data Analysis

**Research Question:**  
*"Does a reduction in mortgage rates predict higher house price growth across euro area countries? Panel evidence 2003-2023."*

**Causal chain tested (one link):** Lower mortgage rates → Higher house prices

---

## Setup (run once)

```bash
pip install fredapi eurostat linearmodels pandas numpy matplotlib seaborn requests openpyxl
```

**FRED API key:** free at [https://fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html)

Set your key in **Section 0** below before running.

In [1]:
# =============================================================================
# SECTION 0 — IMPORTS & CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import requests
import warnings
from io import StringIO
warnings.filterwarnings('ignore')

# ── SET YOUR FREE FRED API KEY HERE ──────────────────────────────────────────
# Get one instantly at: https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY = os.environ.get('FRED_API_KEY', 'YOUR_FRED_API_KEY_HERE')

# ── STUDY PARAMETERS ─────────────────────────────────────────────────────────
START = '2003-01-01'   # Pre-QE baseline starts here
END   = '2023-12-31'   # Covers full APP era and tightening cycle
QE_START = '2015-01-01'  # ECB APP launch — used to split sample later

# ── 9 EURO AREA COUNTRIES ────────────────────────────────────────────────────
# Chosen for data completeness. Greece/Cyprus excluded (structural breaks).
COUNTRIES = {
    'DE': 'Germany',
    'FR': 'France',
    'IT': 'Italy',
    'ES': 'Spain',
    'NL': 'Netherlands',
    'BE': 'Belgium',
    'AT': 'Austria',
    'PT': 'Portugal',
    'FI': 'Finland',
}

print("=" * 65)
print("  EURO AREA HOUSING & MONETARY POLICY — PANEL DATA ANALYSIS")
print("=" * 65)

  EURO AREA HOUSING & MONETARY POLICY — PANEL DATA ANALYSIS


## Section 1 — Data Collection

We pull 5 datasets from 2 free sources: **FRED** and **ECB SDW** (plus Eurostat for controls). Each dataset becomes one column in the final panel.

In [2]:
# ── 1A: HOUSE PRICE INDEX from FRED (BIS Residential Property Prices) ────────
print("\n[1A] Pulling House Price Indices from FRED (BIS series)...")

# BIS Residential Property Price series codes on FRED
# Pattern: Q + {country ISO} + N628BIS  (N = nominal, Index 2010=100)
# NOTE: N368BIS = annual % growth (NOT levels) — do NOT pct_change those!
FRED_HPI = {
    'DE': 'QDEN628BIS',
    'FR': 'QFRN628BIS',
    'IT': 'QITN628BIS',
    'ES': 'QESN628BIS',
    'NL': 'QNLN628BIS',
    'BE': 'QBEN628BIS',
    'AT': 'QATN628BIS',
    'PT': 'QPTN628BIS',
    'FI': 'QFIN628BIS',
}

def fetch_fred_series(series_id, api_key, start, end):
    """Fetches a single time series from FRED via their REST API."""
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id':        series_id,
        'api_key':          api_key,
        'file_type':        'json',
        'observation_start': start,
        'observation_end':   end,
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()['observations']
        df = pd.DataFrame(data)
        df['date']  = pd.to_datetime(df['date'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        return df.set_index('date')['value'].dropna()
    except Exception as e:
        print(f"  WARNING: Could not fetch {series_id} — {e}")
        return pd.Series(dtype=float)

hpi_list = []
for code, series_id in FRED_HPI.items():
    s = fetch_fred_series(series_id, FRED_API_KEY, START, END)
    if not s.empty:
        tmp = s.resample('QE').last().to_frame('hpi')
        tmp['country'] = code
        hpi_list.append(tmp)
        print(f"  {COUNTRIES[code]}: {len(s)} obs")

df_hpi = pd.concat(hpi_list).reset_index()
df_hpi.columns = ['date', 'hpi', 'country']
print(f"  Total HPI rows: {len(df_hpi)}")


[1A] Pulling House Price Indices from FRED (BIS series)...


  Germany: 84 obs


  France: 84 obs


  Italy: 84 obs


  Spain: 84 obs


  Netherlands: 84 obs


  Belgium: 84 obs


  Austria: 84 obs


  Portugal: 84 obs


  Finland: 84 obs
  Total HPI rows: 756


In [3]:
# ── 1B: MORTGAGE RATES from ECB SDW (via REST API) ───────────────────────────
print("\n[1B] Pulling Mortgage Rates from ECB SDW API...")

ECB_COUNTRIES = '+'.join(COUNTRIES.keys())
ECB_SERIES = f'M.{ECB_COUNTRIES}.B.A2C.AM.R.A.2250.EUR.N'

ecb_url = f'https://data-api.ecb.europa.eu/service/data/MIR/{ECB_SERIES}'
ecb_params = {
    'format':      'csvdata',
    'startPeriod': '2003-01',
    'endPeriod':   '2023-12',
    'detail':      'dataonly',
}

try:
    ecb_r = requests.get(ecb_url, params=ecb_params, timeout=30)
    ecb_r.raise_for_status()
    df_ecb_raw = pd.read_csv(StringIO(ecb_r.text))

    df_mort = (df_ecb_raw[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']]
               .rename(columns={'REF_AREA': 'country',
                                'TIME_PERIOD': 'date',
                                'OBS_VALUE': 'mortgage_rate'})
               .copy())
    df_mort['date']          = pd.to_datetime(df_mort['date'], format='%Y-%m')
    df_mort['mortgage_rate'] = pd.to_numeric(df_mort['mortgage_rate'], errors='coerce')

    df_mort = (df_mort.groupby(['country', pd.Grouper(key='date', freq='QE')])
                      ['mortgage_rate'].mean()
                      .reset_index())
    print(f"  Mortgage rate data: {len(df_mort)} country-quarter observations")

except Exception as e:
    print(f"  WARNING: ECB API call failed — {e}")
    print("  Tip: Check your internet connection or try again later.")
    df_mort = pd.DataFrame(columns=['country', 'date', 'mortgage_rate'])


[1B] Pulling Mortgage Rates from ECB SDW API...


  Mortgage rate data: 756 country-quarter observations


In [4]:
# ── 1C: ECB POLICY RATE from FRED ────────────────────────────────────────────
print("\n[1C] Pulling ECB Policy Rate from FRED...")

ecb_rate = fetch_fred_series('ECBDFR', FRED_API_KEY, START, END)
if not ecb_rate.empty:
    df_ecb_rate = (ecb_rate.resample('QE').mean()
                            .to_frame('ecb_rate')
                            .reset_index())
    df_ecb_rate.columns = ['date', 'ecb_rate']
    print(f"  ECB rate: {len(df_ecb_rate)} quarterly observations")
else:
    df_ecb_rate = pd.DataFrame(columns=['date', 'ecb_rate'])


[1C] Pulling ECB Policy Rate from FRED...


  ECB rate: 84 quarterly observations


In [5]:
# ── 1D: GDP GROWTH, UNEMPLOYMENT, PERMITS from Eurostat ─────────────────────
print("\n[1D] Pulling control variables from Eurostat...")

try:
    import eurostat

    print("  Fetching GDP growth (namq_10_gdp)...")
    gdp_raw = eurostat.get_data_df('namq_10_gdp', flags=False)
    gdp_filtered = gdp_raw[
        (gdp_raw['unit']   == 'PD_PCH_PRE_EUR') &
        (gdp_raw['na_item']== 'B1GQ') &
        (gdp_raw['s_adj']  == 'SCA') &
        (gdp_raw['geo\\TIME_PERIOD'].isin(COUNTRIES.keys()))
    ].copy()
    id_cols = [c for c in gdp_filtered.columns if not c.replace('-','').replace('Q','').isdigit()]
    gdp_long = gdp_filtered.melt(id_vars=id_cols, var_name='period', value_name='gdp_growth')
    gdp_long = gdp_long.rename(columns={'geo\\TIME_PERIOD': 'country'})
    gdp_long['date'] = pd.PeriodIndex(gdp_long['period'], freq='Q').to_timestamp('Q')
    gdp_long['gdp_growth'] = pd.to_numeric(gdp_long['gdp_growth'], errors='coerce')
    df_gdp = gdp_long[['country', 'date', 'gdp_growth']].dropna()
    print(f"  GDP growth: {len(df_gdp)} country-quarter obs")

    print("  Fetching building permits (sts_cobp_q)...")
    permits_raw = eurostat.get_data_df('sts_cobp_q', flags=False)
    permits_filtered = permits_raw[
        (permits_raw['s_adj'] == 'SCA') &
        (permits_raw['indic_bt'] == 'BPRM_DW') &
        (permits_raw['cpa2_1'] == 'CPA_F41001_X_410014') &
        (permits_raw['unit'] == 'I15') &
        (permits_raw['geo\\TIME_PERIOD'].isin(COUNTRIES.keys()))
    ].copy()
    id_cols_p = [c for c in permits_filtered.columns if not c.replace('-','').replace('Q','').isdigit()]
    permits_long = permits_filtered.melt(id_vars=id_cols_p, var_name='period', value_name='permits')
    permits_long = permits_long.rename(columns={'geo\\TIME_PERIOD': 'country'})
    permits_long['date'] = pd.PeriodIndex(permits_long['period'], freq='Q').to_timestamp('Q')
    permits_long['permits'] = pd.to_numeric(permits_long['permits'], errors='coerce')
    df_permits = permits_long[['country', 'date', 'permits']].dropna()
    print(f"  Building permits: {len(df_permits)} country-quarter obs")

except Exception as e:
    print(f"  WARNING: Eurostat pull failed — {e}")
    df_gdp    = pd.DataFrame(columns=['country', 'date', 'gdp_growth'])
    df_permits= pd.DataFrame(columns=['country', 'date', 'permits'])


[1D] Pulling control variables from Eurostat...
  Fetching GDP growth (namq_10_gdp)...


  GDP growth: 1204 country-quarter obs
  Fetching building permits (sts_cobp_q)...


  Building permits: 877 country-quarter obs


In [6]:
# ── 1E: UNEMPLOYMENT from FRED (full 2003–2023 coverage) ─────────────────────
# Eurostat une_rt_q only starts in 2009 for most countries. FRED OECD harmonised
# quarterly rates (LRHUTTTT*Q156S) cover the full sample for all 9 countries.
print("\n[1E] Pulling Unemployment from FRED (OECD harmonised quarterly)...")

FRED_UNEMP = {
    'DE': 'LRHUTTTTDEQ156S',
    'FR': 'LRHUTTTTFRQ156S',
    'IT': 'LRHUTTTTITQ156S',
    'ES': 'LRHUTTTTESQ156S',
    'NL': 'LRHUTTTTNLQ156S',
    'BE': 'LRHUTTTTBEQ156S',
    'AT': 'LRHUTTTTATQ156S',
    'PT': 'LRHUTTTTPTQ156S',
    'FI': 'LRHUTTTTFIQ156S',
}

unemp_list = []
for code, series_id in FRED_UNEMP.items():
    s = fetch_fred_series(series_id, FRED_API_KEY, START, END)
    if not s.empty:
        tmp = s.to_frame('unemployment').reset_index()
        tmp.columns = ['date', 'unemployment']
        tmp['date'] = pd.to_datetime(tmp['date']).dt.to_period('Q').dt.to_timestamp('Q')
        tmp['country'] = code
        unemp_list.append(tmp)
        print(f"  {COUNTRIES[code]}: {len(s)} obs")

df_unemp = pd.concat(unemp_list).reset_index(drop=True)
print(f"  Total unemployment rows: {len(df_unemp)}")


[1E] Pulling Unemployment from FRED (OECD harmonised quarterly)...


  Germany: 84 obs


  France: 84 obs


  Italy: 84 obs


  Spain: 84 obs


  Netherlands: 84 obs


  Belgium: 84 obs


  Austria: 84 obs


  Portugal: 84 obs


  Finland: 84 obs
  Total unemployment rows: 756


## Section 2 — Build the Panel Dataset

A panel dataset has two dimensions: country (i) and time (t). Each row = one country in one quarter. We merge all datasets on `[country, date]`.

**Missing-value strategy (instead of dropping 190+ rows):**
1. **Better data source** — FRED unemployment replaces Eurostat (which only starts in 2009 for 8/9 countries).
2. **Within-country interpolation** — linear fill for the 7 scattered missing building-permit values.

Other approaches you could use: restrict the sample to 2009+, drop thin controls, or multiple imputation.

In [7]:
print("\n[2] Merging all datasets into panel...")

panel = df_hpi.copy()
panel = panel.merge(df_mort,    on=['country','date'], how='left')
panel = panel.merge(df_gdp,     on=['country','date'], how='left')
panel = panel.merge(df_unemp,   on=['country','date'], how='left')
panel = panel.merge(df_permits, on=['country','date'], how='left')
panel = panel.merge(df_ecb_rate, on=['date'],          how='left')

panel = panel.sort_values(['country', 'date']).reset_index(drop=True)

# ── FILL REMAINING GAPS (avoid listwise deletion in regression) ───────────────
print("\n  Missing values BEFORE fill:")
print(panel[['mortgage_rate','gdp_growth','unemployment','permits']].isnull().sum())

# Interpolate isolated permit gaps within each country time series
panel['permits'] = (panel.groupby('country')['permits']
                      .transform(lambda s: s.interpolate(method='linear', limit_direction='both')))

print("\n  Missing values AFTER fill:")
print(panel[['mortgage_rate','gdp_growth','unemployment','permits']].isnull().sum())

panel = panel.sort_values(['country', 'date']).reset_index(drop=True)
panel['hpi_growth'] = panel.groupby('country')['hpi'].pct_change() * 100
panel['hpi_growth'] = panel['hpi_growth'].replace([np.inf, -np.inf], np.nan)
panel['hpi_growth_yoy'] = panel.groupby('country')['hpi'].pct_change(4) * 100
panel['hpi_growth_yoy'] = panel['hpi_growth_yoy'].replace([np.inf, -np.inf], np.nan)

for k in range(1, 5):
    panel[f'mort_lag{k}'] = panel.groupby('country')['mortgage_rate'].shift(k)

panel['post_qe'] = (panel['date'] >= QE_START).astype(int)

# Phase A regressors: rate changes and QE interaction
panel['mort_change'] = panel.groupby('country')['mortgage_rate'].diff()
panel['mort_change_lag1'] = panel.groupby('country')['mort_change'].shift(1)
panel['mort_x_post_qe'] = panel['mortgage_rate'] * panel['post_qe']

ROBUST_START = '2009-01-01'   # Phase A robustness window (pre-COVID, full controls)
ROBUST_END   = '2019-12-31'

panel = panel.dropna(subset=['hpi_growth'])
panel_idx = panel.set_index(['country', 'date'])

print(f"  Final panel: {panel['country'].nunique()} countries × "
      f"{panel['date'].nunique()} quarters = {len(panel)} observations")
print(f"  Period: {panel['date'].min().date()} to {panel['date'].max().date()}")
print("\n  Missing values by column:")
print(panel[['hpi_growth','mortgage_rate','gdp_growth',
             'unemployment','permits']].isnull().sum())


[2] Merging all datasets into panel...

  Missing values BEFORE fill:
mortgage_rate    0
gdp_growth       0
unemployment     0
permits          7
dtype: int64

  Missing values AFTER fill:
mortgage_rate    0
gdp_growth       0
unemployment     0
permits          0
dtype: int64
  Final panel: 9 countries × 83 quarters = 747 observations
  Period: 2003-06-30 to 2023-12-31

  Missing values by column:
hpi_growth       0
mortgage_rate    0
gdp_growth       0
unemployment     0
permits          0
dtype: int64


## Section 3 — Descriptive Analysis

In [8]:
print("\n[3] Generating descriptive charts...")

plt.style.use('seaborn-v0_8-whitegrid')
BLUE  = '#1F4E79'
RED   = '#C00000'
LBLUE = '#2E75B6'
COLORS = ['#1F4E79','#C00000','#2E75B6','#375623',
          '#7F4F00','#595959','#833C00','#4472C4','#70AD47']

fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
axes = axes.flatten()

for i, (code, name) in enumerate(COUNTRIES.items()):
    data = panel[panel['country'] == code]
    ax   = axes[i]
    ax.bar(data['date'], data['hpi_growth_yoy'],
           color=[RED if v < 0 else LBLUE for v in data['hpi_growth_yoy']],
           width=60, alpha=0.85)
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.axvline(pd.Timestamp(QE_START), color=RED, lw=1.2,
               ls=':', alpha=0.7, label='APP launch')
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_ylabel('YoY %' if i % 3 == 0 else '')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('House Price Growth (Year-on-Year %) — 9 Euro Area Countries\n'
             'Dotted red line = ECB APP launch (Jan 2015)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart1_hpi_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: chart1_hpi_by_country.png")


[3] Generating descriptive charts...


  Saved: chart1_hpi_by_country.png


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg = panel.groupby('date')[['mortgage_rate','hpi_growth_yoy']].mean()
ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(avg.index, avg['mortgage_rate'],
         color=BLUE, lw=2, label='Avg mortgage rate (LHS)')
ax2.plot(avg.index, avg['hpi_growth_yoy'],
         color=RED, lw=2, ls='--', label='Avg HPI growth (RHS)')
ax1.axvline(pd.Timestamp(QE_START), color='gray', lw=1, ls=':')
ax1.set_ylabel('Mortgage Rate (%)', color=BLUE)
ax2.set_ylabel('House Price Growth, YoY (%)', color=RED)
ax1.set_title('Mortgage Rates & House Prices Move Together\n(Euro area average)',
              fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

ax = axes[1]
for i, code in enumerate(COUNTRIES.keys()):
    sub = panel[panel['country'] == code].dropna(subset=['mortgage_rate','hpi_growth'])
    ax.scatter(sub['mortgage_rate'], sub['hpi_growth'],
               label=COUNTRIES[code], color=COLORS[i], alpha=0.5, s=20)

valid = panel.dropna(subset=['mortgage_rate','hpi_growth'])
z = np.polyfit(valid['mortgage_rate'], valid['hpi_growth'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['mortgage_rate'].min(), valid['mortgage_rate'].max(), 100)
ax.plot(x_line, p(x_line), color='black', lw=2, ls='--', label='Trend line')
ax.axhline(0, color='gray', lw=0.6)
ax.set_xlabel('Mortgage Rate (%)')
ax.set_ylabel('House Price Growth, QoQ (%)')
ax.set_title('Lower Rates, Higher Price Growth\n(All country-quarters pooled)',
             fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart2_mortgage_vs_hpi.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: chart2_mortgage_vs_hpi.png")

  Saved: chart2_mortgage_vs_hpi.png


In [10]:
fig, ax = plt.subplots(figsize=(8, 6))
corr_vars = ['hpi_growth', 'mortgage_rate', 'gdp_growth',
             'unemployment', 'permits', 'ecb_rate']
corr_labels = ['HPI Growth', 'Mortgage Rate', 'GDP Growth',
               'Unemployment', 'Permits', 'ECB Rate']
corr_data = panel[corr_vars].dropna().corr()
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=corr_labels, yticklabels=corr_labels,
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Euro Area Panel Variables',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart3_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: chart3_correlation.png")

  Saved: chart3_correlation.png


## Section 4 — Panel Regression

We use **linearmodels** with two-way fixed effects (country + time). Standard errors are clustered by country.

**Original models (QoQ growth):**
- **Model 1:** Baseline — contemporaneous mortgage rate + controls
- **Model 2:** Distributed lag — mortgage rates at t through t-4
- **Model 3:** Sub-period comparison — pre-QE vs QE era (2015–2019)

**Phase A improvements:**
- **Model 4A:** YoY house price growth as dependent variable
- **Model 4B:** Mortgage **rate changes** (Δrate) + 1 lag
- **Model 4C:** **Interaction** — mortgage rate × post-QE (replaces split-sample Model 3)
- **Model 4D:** Robustness — **2009–2019** subsample (YoY baseline)

In [11]:
print("\n[4] Running panel regressions...")

try:
    from linearmodels.panel import PanelOLS
    import statsmodels.api as sm

    data_reg = panel.set_index(['country', 'date']).copy()

    base_vars = ['mortgage_rate', 'gdp_growth', 'unemployment', 'permits']
    data_base = data_reg[['hpi_growth'] + base_vars].replace([np.inf, -np.inf], np.nan).dropna()

    mod1 = PanelOLS(
        dependent     = data_base['hpi_growth'],
        exog          = data_base[base_vars],
        entity_effects = True,
        time_effects   = True,
    )
    res1 = mod1.fit(cov_type='clustered', cluster_entity=True)
    print("\n  === MODEL 1: BASELINE ===")
    print(res1.summary)

    lag_vars  = ['mortgage_rate', 'mort_lag1', 'mort_lag2', 'mort_lag3', 'mort_lag4']
    ctrl_vars = ['gdp_growth', 'unemployment', 'permits']
    all_vars  = lag_vars + ctrl_vars
    data_lag  = data_reg[['hpi_growth'] + all_vars].replace([np.inf, -np.inf], np.nan).dropna()

    mod2 = PanelOLS(
        dependent      = data_lag['hpi_growth'],
        exog           = data_lag[all_vars],
        entity_effects = True,
        time_effects   = True,
    )
    res2 = mod2.fit(cov_type='clustered', cluster_entity=True)
    print("\n  === MODEL 2: DISTRIBUTED LAG ===")
    print(res2.summary)

    cum_effect = sum(res2.params[v] for v in lag_vars)
    print(f"\n  Cumulative 4-quarter effect of -1pp mortgage rate: "
          f"{-cum_effect:.3f}pp of additional house price growth")

    pre_qe  = data_base[data_base.index.get_level_values('date') < QE_START]
    qe_era  = data_base[(data_base.index.get_level_values('date') >= QE_START) &
                        (data_base.index.get_level_values('date') < '2020-01-01')]

    mod3a = PanelOLS(pre_qe['hpi_growth'], pre_qe[base_vars],
                     entity_effects=True, time_effects=True)
    mod3b = PanelOLS(qe_era['hpi_growth'], qe_era[base_vars],
                     entity_effects=True, time_effects=True)
    res3a = mod3a.fit(cov_type='clustered', cluster_entity=True)
    res3b = mod3b.fit(cov_type='clustered', cluster_entity=True)

    coef_pre = res3a.params.get('mortgage_rate', np.nan)
    coef_qe  = res3b.params.get('mortgage_rate', np.nan)
    print(f"\n  Mortgage rate coefficient:")
    print(f"    Pre-QE  (2003-2014): {coef_pre:.3f}")
    print(f"    QE era  (2015-2019): {coef_qe:.3f}")
    print(f"    Change:              {coef_qe - coef_pre:+.3f}  "
          f"({'stronger' if coef_qe < coef_pre else 'weaker'} during QE)")

    # ==========================================================================
    # PHASE A — IMPROVED SPECIFICATIONS
    # ==========================================================================
    print("\n" + "=" * 65)
    print("  PHASE A — IMPROVED SPECIFICATIONS")
    print("=" * 65)

    ctrl_vars = ['gdp_growth', 'unemployment', 'permits']
    phase_a_rows = []

    def _record(label, res, n, key_vars):
        row = {'Model': label, 'N': n}
        for v in key_vars:
            row[f'{v} coef'] = res.params.get(v, np.nan)
            row[f'{v} p'] = res.pvalues.get(v, np.nan)
        phase_a_rows.append(row)
        return row

    # ── Model 4A: YoY house price growth ─────────────────────────────────────
    yoy_vars = ['mortgage_rate'] + ctrl_vars
    data_yoy = data_reg[['hpi_growth_yoy'] + yoy_vars].replace([np.inf, -np.inf], np.nan).dropna()
    mod4a = PanelOLS(data_yoy['hpi_growth_yoy'], data_yoy[yoy_vars],
                     entity_effects=True, time_effects=True)
    res4a = mod4a.fit(cov_type='clustered', cluster_entity=True)
    print(f"\n  === MODEL 4A: YoY BASELINE (N={len(data_yoy)}) ===")
    print(res4a.summary)
    _record('4A YoY baseline', res4a, len(data_yoy), ['mortgage_rate'])

    # ── Model 4B: Δ mortgage rate + 1 lag ────────────────────────────────────
    dvars = ['mort_change', 'mort_change_lag1'] + ctrl_vars
    data_drate = data_reg[['hpi_growth_yoy'] + dvars].replace([np.inf, -np.inf], np.nan).dropna()
    mod4b = PanelOLS(data_drate['hpi_growth_yoy'], data_drate[dvars],
                     entity_effects=True, time_effects=True)
    res4b = mod4b.fit(cov_type='clustered', cluster_entity=True)
    print(f"\n  === MODEL 4B: Δ MORTGAGE RATE + 1 LAG (N={len(data_drate)}) ===")
    print(res4b.summary)
    cum_d = res4b.params.get('mort_change', 0) + res4b.params.get('mort_change_lag1', 0)
    print(f"\n  Cumulative effect of -1pp rate change (t + t-1): {-cum_d:.3f}pp YoY HPI growth")
    _record('4B Δ rate + lag', res4b, len(data_drate), ['mort_change', 'mort_change_lag1'])

    # ── Model 4C: mortgage rate × post-QE interaction ───────────────────────
    int_vars = ['mortgage_rate', 'mort_x_post_qe'] + ctrl_vars
    data_int = data_reg[['hpi_growth_yoy'] + int_vars].replace([np.inf, -np.inf], np.nan).dropna()
    mod4c = PanelOLS(data_int['hpi_growth_yoy'], data_int[int_vars],
                     entity_effects=True, time_effects=True)
    res4c = mod4c.fit(cov_type='clustered', cluster_entity=True)
    print(f"\n  === MODEL 4C: QE INTERACTION (N={len(data_int)}) ===")
    print(res4c.summary)
    b_pre  = res4c.params.get('mortgage_rate', np.nan)
    b_int  = res4c.params.get('mort_x_post_qe', np.nan)
    print(f"\n  Implied mortgage coef pre-QE (2015):  {b_pre:.3f}")
    print(f"  Implied mortgage coef post-QE (2015+): {b_pre + b_int:.3f}")
    print(f"  Interaction p-value:                   {res4c.pvalues.get('mort_x_post_qe', np.nan):.4f}")
    _record('4C QE interaction', res4c, len(data_int),
            ['mortgage_rate', 'mort_x_post_qe'])

    # ── Model 4D: robustness 2009–2019 ───────────────────────────────────────
    dates = data_reg.index.get_level_values('date')
    mask_robust = (dates >= ROBUST_START) & (dates <= ROBUST_END)
    data_robust = data_reg.loc[mask_robust, ['hpi_growth_yoy'] + yoy_vars]
    data_robust = data_robust.replace([np.inf, -np.inf], np.nan).dropna()
    mod4d = PanelOLS(data_robust['hpi_growth_yoy'], data_robust[yoy_vars],
                     entity_effects=True, time_effects=True)
    res4d = mod4d.fit(cov_type='clustered', cluster_entity=True)
    print(f"\n  === MODEL 4D: YoY BASELINE 2009–2019 (N={len(data_robust)}) ===")
    print(res4d.summary)
    _record('4D 2009-2019 YoY', res4d, len(data_robust), ['mortgage_rate'])

    phase_a_summary = pd.DataFrame(phase_a_rows)
    print("\n  === PHASE A COMPARISON TABLE ===")
    print(phase_a_summary.to_string(index=False))
    phase_a_summary.to_csv('phase_a_results.csv', index=False)
    print("  Saved: phase_a_results.csv")

except ImportError:
    print("  WARNING: linearmodels not installed.")
    print("  Run: pip install linearmodels")
except Exception as e:
    print(f"  WARNING: Regression failed — {e}")


[4] Running panel regressions...



  === MODEL 1: BASELINE ===
                          PanelOLS Estimation Summary                           
Dep. Variable:             hpi_growth   R-squared:                        0.1662
Estimator:                   PanelOLS   R-squared (Between):             -44.454
No. Observations:                 747   R-squared (Within):               0.0204
Date:                Sat, Jun 06 2026   R-squared (Overall):             -8.6119
Time:                        09:28:30   Log-likelihood                   -1227.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      32.493
Entities:                           9   P-value                           0.0000
Avg Obs:                       83.000   Distribution:                   F(4,652)
Min Obs:                       83.000                                           
Max Obs:                       83.000   F-statistic (robust):             93.891

## Section 5 — Visualise Results

In [12]:
print("\n[5] Generating results charts...")

try:
    coefs = [res2.params.get(v, np.nan) for v in lag_vars]
    ses   = [res2.std_errors.get(v, np.nan) for v in lag_vars]
    cumulative  = np.nancumsum(coefs)
    cum_se_approx = np.nancumsum(ses)
    ci_low  = cumulative - 1.96 * cum_se_approx
    ci_high = cumulative + 1.96 * cum_se_approx

    fig, ax = plt.subplots(figsize=(10, 5))
    horizons = range(5)
    ax.plot(horizons, cumulative, 'o-', color=BLUE, lw=2.5,
            markersize=8, label='Cumulative effect of -1pp mortgage rate')
    ax.fill_between(horizons, ci_low, ci_high,
                    alpha=0.15, color=BLUE, label='95% confidence band')
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xticks(horizons)
    ax.set_xticklabels(['Same\nquarter', '+1Q', '+2Q', '+3Q', '+4Q'])
    ax.set_xlabel('Quarters after mortgage rate change', fontsize=12)
    ax.set_ylabel('Cumulative effect on house price growth (pp)', fontsize=12)
    ax.set_title('How Long Does a Mortgage Rate Cut Take to Lift House Prices?\n'
                 'Cumulative response to a 1pp mortgage rate reduction',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('chart4_impulse_response.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  Saved: chart4_impulse_response.png")

except NameError:
    print("  Skipped: run Section 4 regressions first.")
except Exception as e:
    print(f"  WARNING: Chart 4 failed — {e}")


[5] Generating results charts...
  Saved: chart4_impulse_response.png


In [13]:
try:
    fig, ax = plt.subplots(figsize=(8, 5))
    periods = ['Pre-QE\n(2003-2014)', 'QE Era\n(2015-2019)']
    coefs_plot = [coef_pre, coef_qe]
    colors_bar = [LBLUE, RED]
    bars = ax.bar(periods, coefs_plot, color=colors_bar, width=0.4, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Mortgage rate coefficient on HPI growth')
    ax.set_title('Did QE Amplify the Mortgage Rate → House Price Channel?\n'
                 'Panel FE coefficient on mortgage rate, by sub-period',
                 fontsize=12, fontweight='bold')
    for bar, val in zip(bars, coefs_plot):
        ax.text(bar.get_x() + bar.get_width()/2, val - 0.05,
                f'{val:.3f}', ha='center', va='top',
                fontweight='bold', fontsize=13, color='white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('chart5_subperiod.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  Saved: chart5_subperiod.png")

except NameError:
    print("  Skipped: run Section 4 regressions first.")
except Exception as e:
    print(f"  WARNING: Chart 5 failed — {e}")

  Saved: chart5_subperiod.png


In [14]:
try:
    results_summary = pd.DataFrame({
        'Variable': ['Mortgage rate (t)',
                     'Mortgage rate (t-1)',
                     'Mortgage rate (t-2)',
                     'Mortgage rate (t-3)',
                     'Mortgage rate (t-4)',
                     'Cumulative (0 to 4)',
                     'GDP growth',
                     'Unemployment',
                     'Building permits'],
        'Baseline':    ([f"{res1.params.get('mortgage_rate',np.nan):.3f}"] +
                        ['—'] * 4 +
                        [f"{res1.params.get('mortgage_rate',np.nan):.3f}"] +
                        [f"{res1.params.get(v,np.nan):.3f}"
                         for v in ['gdp_growth','unemployment','permits']]),
        'Dist. Lags':  ([f"{res2.params.get(v,np.nan):.3f}" for v in lag_vars] +
                        [f"{cum_effect:.3f}"] +
                        [f"{res2.params.get(v,np.nan):.3f}"
                         for v in ['gdp_growth','unemployment','permits']]),
    })

    print("\n  === RESULTS SUMMARY TABLE ===")
    print(results_summary.to_string(index=False))
    results_summary.to_csv('results_table.csv', index=False)
    print("  Saved: results_table.csv")

except NameError:
    print("  Skipped: run Section 4 regressions first.")
except Exception as e:
    print(f"  WARNING: Results table failed — {e}")


  === RESULTS SUMMARY TABLE ===
           Variable Baseline Dist. Lags
  Mortgage rate (t)   -0.732     -1.079
Mortgage rate (t-1)        —     -0.306
Mortgage rate (t-2)        —      1.080
Mortgage rate (t-3)        —     -0.760
Mortgage rate (t-4)        —      0.550
Cumulative (0 to 4)   -0.732     -0.516
         GDP growth    0.164      0.108
       Unemployment   -0.271     -0.275
   Building permits   -0.001     -0.001
  Saved: results_table.csv


## Section 6 — Export Clean Panel Data

In [15]:
print("\n[6] Saving clean panel dataset...")
panel.to_csv('euro_area_panel.csv', index=False)
print("  Saved: euro_area_panel.csv")
print("\n  Done! Files created:")
print("  euro_area_panel.csv       — clean panel dataset")
print("  results_table.csv         — regression results")
print("  phase_a_results.csv       — Phase A model comparison")
print("  chart1_hpi_by_country.png — HPI time series")
print("  chart2_mortgage_vs_hpi.png — key relationship chart")
print("  chart3_correlation.png    — correlation heatmap")
print("  chart4_impulse_response.png — cumulative effect chart")
print("  chart5_subperiod.png      — pre vs post QE comparison")
print("\n" + "=" * 65)
print("  ANALYSIS COMPLETE")
print("=" * 65)


[6] Saving clean panel dataset...
  Saved: euro_area_panel.csv

  Done! Files created:
  euro_area_panel.csv       — clean panel dataset
  results_table.csv         — regression results
  phase_a_results.csv       — Phase A model comparison
  chart1_hpi_by_country.png — HPI time series
  chart2_mortgage_vs_hpi.png — key relationship chart
  chart3_correlation.png    — correlation heatmap
  chart4_impulse_response.png — cumulative effect chart
  chart5_subperiod.png      — pre vs post QE comparison

  ANALYSIS COMPLETE
